# Object Detection using Faster RCNN (Pre - Trained Model)

## Step 1 -Import Libraries

In [9]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from PIL import Image


# Step 2 -Load Pretrained Faster R-CNN model from Tensorflow Hub


In [10]:
MODEL_URL = "https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_640x640/1"
model = hub.load(MODEL_URL)

# Class labels for COCO Dataset(used by the pretrained model)
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane',
]

# Step 3 -Function for performing object detection
d

In [11]:
def detect_objects(image_path, model, threshold=0.5):
    # Debug message to verify the file path
    print(f"Loading image from: {image_path}")
    
    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"could not load image at {image_path}. Please check the file path")
    # convert RGB format for visualization and tensorflow processing
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Resize the image
    input_tensor = tf.convert_to_tensor(image_rgb , dtype = tf.unit8)
    input_tensor = input_tensor[tf.newaxis, ...] # Add batch dimensions
    
    # Run object detection
    detections = model(input_tensor)
    
    # Extract detection results
    boxes = detections['detection_boxes'][0].numpy()
    class_indices = detections['detection_classes'][0].numpy().astype(int) -1 # class indices,-1 to adjust for zero-indexed
    scores = detections['detection_score'][0].numpy()# confidencce score
    # Filter out detections below the confidence threshold
    valid_detections = scores >= threshold
    boxes = boxes[valid_detections]
    class_indices = class_indices[valid_detections]
    scores = scores[valid_detections]
    return image_rgb, boxes, class_indices, scores

# Step 4: Function for displaying image with bound box

Develop a function for displaying image with bounding box

In [12]:
def display_image_with_detections(image, boxes, class_indices, score, threshold = 0.5):
    height, width,_ = image.shape
    for i, box in enumerate(boxes):
        if score[i] > threshold:
            ymin, xmin, ymax, xmax = box
            (left, right , top, bottom) = (int(xmin * width), int(xmax * width), int(ymin*height), int(ymax*height))
            
            # Draw bounding box
            cv2.rectangle(image, (left, top),(right, bottom), (0.255,0),2)
            
            # Display label and confidence
            label = f"{COCO_CLASSES[class_indices[i]]}:{scores[i]:.2f}"
            cv2.putText(image, label, (left, top -10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
    # show the result with matplotlib
    plt.figure(figsize=(15,15))
    plt.imshow(image)
    plt.axis("off")
    plt.show()
    


# Step 6 -Evaluating the Model with the given image

In [13]:
image_path = "./images/Lion.jpeg"
try:
    image_rgb, detected_boxes, detected_classes, detected_scores = detect_objects(image_path, model , threshold = 0.5)
    display_image_with_detections(image_rgb , detected_boxes, detected_classes , detected_scores)
except FileNotFoundError:
    print(f"could not load image at {image_path}. Please check the file path")

Loading image from: ./images/Lion.jpeg


AttributeError: module 'tensorflow' has no attribute 'unit8'